In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.utils as vutils

import matplotlib.pyplot as plt
import numpy as np
import os
import time

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# set seeds so results are reproducible
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
LATENT_DIM = 100       # size of noise vector z
IMG_SIZE = 64          # images will be 64x64
CHANNELS = 3           # RGB
BATCH_SIZE = 64
LR = 0.0002
BETA1 = 0.5
BETA2 = 0.999
NUM_EPOCHS_DCGAN = 50
NUM_EPOCHS_WGAN = 50
LAMBDA_GP = 10         # gradient penalty weight
CRITIC_ITERS = 5       # how many times critic trains per generator step
SAVE_INTERVAL = 10     # save checkpoint every N epochs

In [ ]:
data_path = "/kaggle/input/datasets/soumikrakshit/anime-faces"  # adjust this if needed

# transforms: resize, crop to 64x64, convert to tensor, normalize to [-1,1]
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

dataset = datasets.ImageFolder(root=data_path, transform=transform)
print(f"Total images in dataset: {len(dataset)}")

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    drop_last=True
)

# quick check - grab one batch and see the shape
sample_batch = next(iter(dataloader))
print("Batch shape:", sample_batch[0].shape)  # should be [64, 3, 64, 64]

In [ ]:
def show_images(images, title="", num=16):
    # denormalize from [-1,1] to [0,1]
    images = images * 0.5 + 0.5
    grid = vutils.make_grid(images[:num], nrow=4, padding=2)
    plt.figure(figsize=(8, 8))
    plt.title(title)
    plt.axis("off")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.show()

real_images, _ = next(iter(dataloader))
show_images(real_images, "Real Anime Faces")

In [ ]:
def init_weights(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim, channels):
        super(Generator, self).__init__()
        
        self.main = nn.Sequential(
            # layer 1: (latent_dim) x 1 x 1 -> 512 x 4 x 4
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            
            # layer 2: 512 x 4 x 4 -> 256 x 8 x 8
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            # layer 3: 256 x 8 x 8 -> 128 x 16 x 16
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            # layer 4: 128 x 16 x 16 -> 64 x 32 x 32
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            # output layer: 64 x 32 x 32 -> 3 x 64 x 64
            nn.ConvTranspose2d(64, channels, 4, 2, 1, bias=False),
            nn.Tanh()
            # tanh gives output in [-1, 1] which matches our normalization
        )
    
    def forward(self, x):
        return self.main(x)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels):
        super(Discriminator, self).__init__()
        
        self.main = nn.Sequential(
            # layer 1: 3 x 64 x 64 -> 64 x 32 x 32
            # no batchnorm on first layer (DCGAN paper)
            nn.Conv2d(channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # layer 2: 64 x 32 x 32 -> 128 x 16 x 16
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            # layer 3: 128 x 16 x 16 -> 256 x 8 x 8
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # layer 4: 256 x 8 x 8 -> 512 x 4 x 4
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            # output: 512 x 4 x 4 -> 1 x 1 x 1
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )
    
    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)
        # squeeze so output is (batch,) not (batch, 1, 1, 1)

In [ ]:
class Critic(nn.Module):
    def __init__(self, channels):
        super(Critic, self).__init__()
        
        self.main = nn.Sequential(
            # layer 1: no normalization
            nn.Conv2d(channels, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # layer 2
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # layer 3
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # layer 4
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # output - NO sigmoid here, just raw score
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )
    
    def forward(self, x):
        return self.main(x).view(-1, 1).squeeze(1)

In [ ]:
def compute_gradient_penalty(critic, real_imgs, fake_imgs, device):
    batch_size = real_imgs.size(0)
    
    # random interpolation factor between real and fake
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    
    # create interpolated images
    interpolated = (alpha * real_imgs + (1 - alpha) * fake_imgs)
    interpolated = interpolated.requires_grad_(True)
    
    # get critic score on interpolated
    critic_interpolated = critic(interpolated)
    
    # compute gradients of critic score wrt interpolated images
    gradients = torch.autograd.grad(
        outputs=critic_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interpolated),
        create_graph=True,   # need this to backprop through the gradient
        retain_graph=True
    )[0]
    
    # flatten gradients to (batch_size, -1) then compute norm
    gradients = gradients.view(batch_size, -1)
    gradient_norm = torch.sqrt(torch.sum(gradients ** 2, dim=1) + 1e-12)
    
    # penalty is (norm - 1)^2 averaged over batch
    penalty = torch.mean((gradient_norm - 1.0) ** 2)
    return penalty

In [ ]:
netG = Generator(LATENT_DIM, CHANNELS).to(device)
netG.apply(init_weights)

netD = Discriminator(CHANNELS).to(device)
netD.apply(init_weights)

# sanity check - make sure shapes work
test_noise = torch.randn(4, LATENT_DIM, 1, 1, device=device)
test_fake = netG(test_noise)
print("Generator output shape:", test_fake.shape)  # should be [4, 3, 64, 64]

test_pred = netD(test_fake)
print("Discriminator output shape:", test_pred.shape)  # should be [4]
print("Discriminator output values:", test_pred.data)

print(f"\nGenerator params: {sum(p.numel() for p in netG.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in netD.parameters()):,}")


In [ ]:
dcgan_G = Generator(LATENT_DIM, CHANNELS).to(device)
dcgan_G.apply(init_weights)
dcgan_D = Discriminator(CHANNELS).to(device)
dcgan_D.apply(init_weights)

# loss and optimizers
bce_loss = nn.BCEWithLogitsLoss()
opt_G = optim.Adam(dcgan_G.parameters(), lr=LR, betas=(BETA1, BETA2))
opt_D = optim.Adam(dcgan_D.parameters(), lr=LR, betas=(BETA1, BETA2))

# mixed precision
scaler_G = GradScaler()
scaler_D = GradScaler()

# fixed noise to track progress visually
fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

# loss tracking
dcgan_g_losses = []
dcgan_d_losses = []
dcgan_samples = []  # store generated images at intervals

print("Starting DCGAN training...")
start_time = time.time()

for epoch in range(NUM_EPOCHS_DCGAN):
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    num_batches = 0
    
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)
        
        # labels - using 0.9 for real (label smoothing helps stability)
        real_labels = torch.full((batch_size,), 0.9, device=device)
        fake_labels = torch.zeros(batch_size, device=device)
        
        # =====================
        # Train Discriminator
        # =====================
        opt_D.zero_grad()
        
        with autocast():
            # discriminator on real images
            output_real = dcgan_D(real_imgs)
            loss_real = bce_loss(output_real, real_labels)
            
            # generate fakes
            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_imgs = dcgan_G(noise)
            
            # discriminator on fake images
            # .detach() so gradients dont flow back to generator here
            output_fake = dcgan_D(fake_imgs.detach())
            loss_fake = bce_loss(output_fake, fake_labels)
            
            d_loss = loss_real + loss_fake
        
        scaler_D.scale(d_loss).backward()
        scaler_D.step(opt_D)
        scaler_D.update()
        
        # =====================
        # Train Generator
        # =====================
        opt_G.zero_grad()
        
        with autocast():
            # we want discriminator to think fakes are real
            output = dcgan_D(fake_imgs)
            g_loss = bce_loss(output, torch.ones(batch_size, device=device))
        
        scaler_G.scale(g_loss).backward()
        scaler_G.step(opt_G)
        scaler_G.update()
        
        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()
        num_batches += 1
    
    # average losses for this epoch
    avg_g = epoch_g_loss / num_batches
    avg_d = epoch_d_loss / num_batches
    dcgan_g_losses.append(avg_g)
    dcgan_d_losses.append(avg_d)
    
    elapsed = time.time() - start_time
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS_DCGAN}]  "
          f"D_loss: {avg_d:.4f}  G_loss: {avg_g:.4f}  "
          f"Time: {elapsed:.0f}s")
    
    # save sample images every few epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        with torch.no_grad():
            sample = dcgan_G(fixed_noise).detach().cpu()
        dcgan_samples.append((epoch + 1, sample))
        show_images(sample, f"DCGAN - Epoch {epoch+1}")
    
    # save checkpoint
    if (epoch + 1) % SAVE_INTERVAL == 0:
        torch.save({
            'epoch': epoch + 1,
            'generator': dcgan_G.state_dict(),
            'discriminator': dcgan_D.state_dict(),
            'opt_g': opt_G.state_dict(),
            'opt_d': opt_D.state_dict(),
        }, f'dcgan_checkpoint_epoch{epoch+1}.pth')
        print(f"  -> Saved checkpoint at epoch {epoch+1}")

print(f"\nDCGAN training finished in {time.time()-start_time:.0f} seconds")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(dcgan_g_losses, label="Generator")
plt.plot(dcgan_d_losses, label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# continue DCGAN training for 10 more epochs
EXTRA_EPOCHS = 10
start_epoch = 50  # where we left off

scaler_G = GradScaler()
scaler_D = GradScaler()

print("Resuming DCGAN training from epoch 50...")
start_time = time.time()

for epoch in range(EXTRA_EPOCHS):
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    num_batches = 0
    
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)
        
        real_labels = torch.full((batch_size,), 0.9, device=device)
        fake_labels = torch.zeros(batch_size, device=device)
        
        # train discriminator
        opt_D.zero_grad()
        with autocast():
            output_real = dcgan_D(real_imgs)
            loss_real = bce_loss(output_real, real_labels)
            
            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_imgs = dcgan_G(noise)
            output_fake = dcgan_D(fake_imgs.detach())
            loss_fake = bce_loss(output_fake, fake_labels)
            d_loss = loss_real + loss_fake
        
        scaler_D.scale(d_loss).backward()
        scaler_D.step(opt_D)
        scaler_D.update()
        
        # train generator
        opt_G.zero_grad()
        with autocast():
            output = dcgan_D(fake_imgs)
            g_loss = bce_loss(output, torch.ones(batch_size, device=device))
        
        scaler_G.scale(g_loss).backward()
        scaler_G.step(opt_G)
        scaler_G.update()
        
        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()
        num_batches += 1
    
    avg_g = epoch_g_loss / num_batches
    avg_d = epoch_d_loss / num_batches
    dcgan_g_losses.append(avg_g)
    dcgan_d_losses.append(avg_d)
    
    current_epoch = start_epoch + epoch + 1
    elapsed = time.time() - start_time
    print(f"Epoch [{current_epoch}/60]  D_loss: {avg_d:.4f}  G_loss: {avg_g:.4f}  Time: {elapsed:.0f}s")
    
    if (epoch + 1) % 5 == 0:
        with torch.no_grad():
            sample = dcgan_G(fixed_noise).detach().cpu()
        dcgan_samples.append((current_epoch, sample))
        show_images(sample, f"DCGAN - Epoch {current_epoch}")

print("extra epochs complete.")

In [ ]:
# save DCGAN checkpoint at epoch 60
torch.save({
    'epoch': 60,
    'generator': dcgan_G.state_dict(),
    'discriminator': dcgan_D.state_dict(),
    'opt_g': opt_G.state_dict(),
    'opt_d': opt_D.state_dict(),
}, 'dcgan_checkpoint_epoch60.pth')

# also save the final generator standalone (for gradio app later)
torch.save(dcgan_G.state_dict(), 'dcgan_generator_final.pth')

print("Saved!")


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(dcgan_g_losses, label="Generator")
plt.plot(dcgan_d_losses, label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# WGAN-GP training - fixed version
wgan_G = Generator(LATENT_DIM, CHANNELS).to(device)
wgan_G.apply(init_weights)
wgan_C = Critic(CHANNELS).to(device)
wgan_C.apply(init_weights)

# key change: lower LR for critic, use separate LRs
opt_wG = optim.Adam(wgan_G.parameters(), lr=1e-4, betas=(0.0, 0.9))
opt_wC = optim.Adam(wgan_C.parameters(), lr=1e-4, betas=(0.0, 0.9))

fixed_noise_w = torch.randn(64, LATENT_DIM, 1, 1, device=device)

wgan_g_losses = []
wgan_c_losses = []
wgan_samples = []

NUM_EPOCHS_WGAN = 50

print("Starting training...")
start_time = time.time()

for epoch in range(NUM_EPOCHS_WGAN):
    epoch_g_loss = 0.0
    epoch_c_loss = 0.0
    num_batches = 0
    
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)
        
        # train critic 5 times
        for _ in range(CRITIC_ITERS):
            opt_wC.zero_grad()
            
            score_real = wgan_C(real_imgs)
            
            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_imgs = wgan_G(noise).detach()
            
            score_fake = wgan_C(fake_imgs)
            
            gp = compute_gradient_penalty(wgan_C, real_imgs, fake_imgs, device)
            
            c_loss = score_fake.mean() - score_real.mean() + LAMBDA_GP * gp
            
            c_loss.backward()
            opt_wC.step()
        
        epoch_c_loss += c_loss.item()
        
        # train generator
        opt_wG.zero_grad()
        
        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
        fake_imgs = wgan_G(noise)
        score = wgan_C(fake_imgs)
        
        g_loss = -score.mean()
        
        g_loss.backward()
        opt_wG.step()
        
        epoch_g_loss += g_loss.item()
        num_batches += 1
    
    avg_g = epoch_g_loss / num_batches
    avg_c = epoch_c_loss / num_batches
    wgan_g_losses.append(avg_g)
    wgan_c_losses.append(avg_c)
    
    elapsed = time.time() - start_time
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS_WGAN}]  "
          f"C_loss: {avg_c:.4f}  G_loss: {avg_g:.4f}  "
          f"Time: {elapsed:.0f}s")
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        with torch.no_grad():
            sample = wgan_G(fixed_noise_w).detach().cpu()
        wgan_samples.append((epoch + 1, sample))
        show_images(sample, f"WGAN-GP - Epoch {epoch+1}")
    
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch + 1,
            'generator': wgan_G.state_dict(),
            'critic': wgan_C.state_dict(),
            'opt_g': opt_wG.state_dict(),
            'opt_c': opt_wC.state_dict(),
        }, f'wgan_checkpoint_epoch{epoch+1}.pth')
        print(f"  -> Saved checkpoint at epoch {epoch+1}")

print(f"\nWGAN-GP training finished in {time.time()-start_time:.0f} seconds")


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(wgan_g_losses, label="Generator")
plt.plot(wgan_c_losses, label="Critic")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("WGAN-GP Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
with torch.no_grad():
    comparison_noise = torch.randn(16, LATENT_DIM, 1, 1, device=device)
    dcgan_output = dcgan_G(comparison_noise).cpu()
    wgan_output = wgan_G(comparison_noise).cpu()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# DCGAN samples
dcgan_grid = vutils.make_grid(dcgan_output * 0.5 + 0.5, nrow=4, padding=2)
axes[0].imshow(dcgan_grid.permute(1, 2, 0).numpy())
axes[0].set_title("DCGAN Generated", fontsize=14)
axes[0].axis("off")

# WGAN-GP samples
wgan_grid = vutils.make_grid(wgan_output * 0.5 + 0.5, nrow=4, padding=2)
axes[1].imshow(wgan_grid.permute(1, 2, 0).numpy())
axes[1].set_title("WGAN-GP Generated", fontsize=14)
axes[1].axis("off")

plt.suptitle("DCGAN vs WGAN-GP Comparison", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
print("=== DCGAN Diversity Check (64 samples) ===")
with torch.no_grad():
    big_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)
    dcgan_diverse = dcgan_G(big_noise).cpu()
show_images(dcgan_diverse, "DCGAN - 64 Samples (check for mode collapse)", num=64)

print("\n=== WGAN-GP Diversity Check (64 samples) ===")
with torch.no_grad():
    wgan_diverse = wgan_G(big_noise).cpu()
show_images(wgan_diverse, "WGAN-GP - 64 Samples (should be more diverse)", num=64)

In [ ]:
def show_training_progress(samples_list, model_name):
    num = len(samples_list)
    fig, axes = plt.subplots(1, num, figsize=(4*num, 4))
    if num == 1:
        axes = [axes]
    
    for idx, (ep, imgs) in enumerate(samples_list):
        grid = vutils.make_grid(imgs[:16] * 0.5 + 0.5, nrow=4, padding=1)
        axes[idx].imshow(grid.permute(1, 2, 0).numpy())
        axes[idx].set_title(f"Epoch {ep}")
        axes[idx].axis("off")
    
    plt.suptitle(f"{model_name} - Training Progress", fontsize=14)
    plt.tight_layout()
    plt.show()

show_training_progress(dcgan_samples, "DCGAN")
show_training_progress(wgan_samples, "WGAN-GP")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(dcgan_g_losses, label="G Loss", color="blue")
axes[0].plot(dcgan_d_losses, label="D Loss", color="orange")
axes[0].set_title("DCGAN Losses")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(wgan_g_losses, label="G Loss", color="blue")
axes[1].plot(wgan_c_losses, label="C Loss", color="orange")
axes[1].set_title("WGAN-GP Losses")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Loss Comparison: DCGAN vs WGAN-GP", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
torch.save(wgan_G.state_dict(), 'wgan_generator_final.pth')
print("Saved final wgan model!")